In [88]:
import os, h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import colors as cols
import numpy as np
from lib import vaspwfc
from pymatgen.core import Lattice, Structure, Molecule
from pymatgen.electronic_structure.core import Spin, Orbital
from pymatgen.io import vasp, ase
import nglview as nv
from skimage import measure
from ase.neighborlist import natural_cutoffs, NeighborList
from scipy.ndimage import gaussian_filter, zoom
from math import pi, exp, sin, cos, log
from ase.data.colors import jmol_colors

In [ ]:
def orbital_image_2D(self):
    for spin_index, spin_name in enumerate(spin_names):
        for k_index in range(n_kpts):
            for array_index, orbital_index in enumerate(orbital_indices):
                wfn2D = wfns[spin_index, k_index, array_index, :, :, z_slice_index]
                density = np.abs(wfn2D) ** 2
                norm_density = density / np.max(density)
                phase = (np.angle(wfn2D) - .25 * pi) % (2 * pi)
                norm_phase = phase / (2 * pi)
                
                orbital_energy = energy_dict["energies"][spin_name][orbital_index]
                
                rgb_img = cols.hsv_to_rgb(np.dstack([norm_phase, np.ones_like(phase), norm_density]))
                #plt.imshow(rgb_img)
                #plt.tight_layout()
                #plt.imsave(f"./{calculation_name}_{spin_name}_kpt{k_index}_orb{orbital_index}@{int(z_slice_height * 10)}A.png", rgb_img)


In [95]:
def find_folder(target: str = "", base_folder = "C:\\DFT") -> str:
    target_folder = None
    sub_folders = [os.path.join(base_folder, folder) for folder in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, folder))]
    for sub_folder in sub_folders:
        if os.path.isdir(os.path.join(sub_folder, target)): target_folder = os.path.join(sub_folder, target)
    if not target_folder:
        for sub_folder in sub_folders:
            subsub_folders = [os.path.join(sub_folder, folder) for folder in os.listdir(sub_folder) if os.path.isdir(os.path.join(sub_folder, folder))]
            for subsub_folder in subsub_folders:
                if os.path.isdir(os.path.join(subsub_folder, target)): target_folder = os.path.join(subsub_folder, target)

    target_folder
    if not target_folder: raise Exception("Folder not found")
    return target_folder

class Initio:
    def __init__(self):
        pass

    def read_vasp_file(self, path: str) -> vaspwfc | Structure | bool:
        base_name = os.path.basename(path)
        
        match base_name:
            case "WAVECAR":
                wfc = self.get_wavecar(path)
                
                return wfc
            case "POSCAR" | "CONTCAR":
                struc = self.get_structure(path)
                return struc
            case _:
                print(f"Could not determine file type of provided file {path}")
                return False

    def get_wavecar(self, path: str) -> vaspwfc:
        try:
            wfc = vaspwfc(path, lgamma = False)
            
            n_kpts = int(wfc._nkpts)
            if n_kpts < 2: wfc = vaspwfc(path, lgamma = True)
            
            return wfc
        except Exception as e:
            print("Error loading the wavecar")
            return False

    def get_structure(self, path: str) -> Structure:
        try:
            structure = Structure.from_file(path)
            return structure
        except Exception as e:
            print(f"Error loading the structure: {e}")
            return False

    def DOS_from_energies(self, eigenenergies: list | np.ndarray = [], gamma = None, sigma = None, energy_range = None, points = None, dE: float = 0.1, weights: list | np.ndarray = []) -> np.ndarray:
        use_weights = False
        
        if isinstance(eigenenergies, list): eigenenergies = np.array(eigenenergies, dtype = float)
        if not isinstance(eigenenergies, np.ndarray): raise TypeError("No valid energy list given")
        
        if isinstance(weights, list): weights = np.array(weights, dtype = float)
        if isinstance(weights, np.ndarray) and len(weights) == len(eigenenergies): use_weights = True

        E_min = np.min(eigenenergies)
        E_max = np.max(eigenenergies)
        
        if isinstance(energy_range, list | np.ndarray):
            energy_range.sort()
            if len(energy_range) > 1:
                E_min = energy_range[0]
                E_max = energy_range[1]
        
        if isinstance(points, int): # Explicit specification of the number of points triggers the energy list to be composed using linspace
            E_list = np.linspace(E_min, E_max, points, dtype = float)
        else: # Use energy spacing instead
            E_list = np.arange(E_min, E_max + dE, dE)
            
        DOS = np.stack([E_list, np.zeros_like(E_list)], dtype = float)
        


        # Use Lorentzian broadening
        if isinstance(gamma, float) and gamma > 0:
            gamma2 = gamma ** 2
            
            for index, energy in enumerate(E_list):
                en_diff_list = eigenenergies - energy
                en_diff_list2 = en_diff_list ** 2
                
                if use_weights:
                    for eigenenergy_index, delta_E2 in enumerate(en_diff_list2):
                        DOS[1, index] += weights[eigenenergy_index] * gamma / (gamma2 + delta_E2)
                else:
                    for delta_E2 in en_diff_list2:
                        DOS[1, index] += gamma / (gamma2 + delta_E2)
        
        return DOS

    def get_HOMO_LUMO(self, wavecar_object: vaspwfc) -> dict:
        eigenstate_dict = self.get_eigenenergies_from_wavecar(wavecar_object)
        
        bands_up = eigenstate_dict["energies"]["spin up"]
        bands_down = eigenstate_dict["energies"]["spin down"]
        occs_up = eigenstate_dict["occupations"]["spin up"]
        occs_down = eigenstate_dict["occupations"]["spin down"]
        
        LUMO_up_index = int(np.where(occs_up < .5)[0][0])
        HOMO_up_index = LUMO_up_index - 1
        LUMO_down_index = int(np.where(occs_down < .5)[0][0])
        HOMO_down_index = LUMO_down_index - 1
        
        HOMO_up_energy = float(bands_up[HOMO_up_index])
        HOMO_down_energy = float(bands_down[HOMO_down_index])
        LUMO_up_energy = float(bands_up[LUMO_up_index])
        LUMO_down_energy = float(bands_down[LUMO_down_index])
        
        return {"HOMO_up_index": HOMO_up_index, "HOMO_down_index": HOMO_down_index, "LUMO_up_index": LUMO_up_index, "LUMO_down_index": LUMO_down_index,
                "HOMO_up_energy": HOMO_up_energy, "HOMO_down_energy": HOMO_down_energy, "LUMO_up_energy": LUMO_up_energy, "LUMO_down_energy": LUMO_down_energy}

    def get_eigenenergies_from_wavecar(self, wavecar_object: vaspwfc) -> dict:
        n_spins = int(wavecar_object._nspin)
        n_kpts = int(wavecar_object._nkpts)
        all_bands = wavecar_object._bands
        all_band_occs = wavecar_object._occs
        
        bands_up = []
        bands_down = []
        occs_up = []
        occs_down = []
        for kpt in range(n_kpts):
            bands_up_k: list = all_bands[0][kpt] # Retrieve bands and occupations at spin index 0 and k-point index kpt
            occs_up_k: list = all_band_occs[0][kpt]
            
            match n_spins:
                case 2: # If spin-polarized, retrieve the spin down energies from spin index 1
                    bands_down_k = all_bands[1][kpt]
                    occs_down_k = all_band_occs[1][kpt]
                case _: # If not spin-polarized, copy the spin up energies to the spin down energies
                    bands_down_k = bands_up_k[:]
                    occs_down_k = occs_up_k[:]
            
            bands_up.extend(bands_up_k)
            bands_down.extend(bands_down_k)
            occs_up.extend(occs_up_k)
            occs_down.extend(occs_down_k)

        bands_up = np.array(bands_up, dtype = np.float32)
        bands_down = np.array(bands_down, dtype = np.float32)
        occs_up = np.array(occs_up, dtype = np.float32)
        occs_down = np.array(occs_down, dtype = np.float32)

        return {"energies": {"spin up": bands_up, "spin down": bands_down}, "occupations": {"spin up": occs_up, "spin down": occs_down}}

    def spin_and_occupation_resolved_DOS(self, wavecar_object: vaspwfc, *args, **kwargs) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        weights = kwargs.pop("weights", None)
        
        energy_dict = self.get_eigenenergies_from_wavecar(wavecar_object)
        [bands_up, bands_down] = [energy_dict["energies"][spin] for spin in ["spin up", "spin down"]]
        [occs_up, occs_down] = [energy_dict["occupations"][spin] for spin in ["spin up", "spin down"]]
        
        LDOS_up_occ = self.DOS_from_energies(bands_up, weights = occs_up, *args, **kwargs)
        LDOS_up_unocc = self.DOS_from_energies(bands_up, weights = 1 - occs_up, *args, **kwargs)
        LDOS_down_occ = self.DOS_from_energies(bands_down, weights = occs_down, *args, **kwargs)
        LDOS_down_unocc = self.DOS_from_energies(bands_down, weights = 1 - occs_down, *args, **kwargs)
        
        return (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc)

    def DOS_plot(self, wavecar_object: vaspwfc, *args, **kwargs) -> plt.Figure:
        colors = kwargs.pop("colors", None)
        
        # No colors given. Use defaults
        if not isinstance(colors, list) or len(colors) < 2: colors = ["#A00000", "#0000A0"]
        # Invalid colors given. Use defaults
        if not cols.is_color_like(colors[0]): colors = ["#A00000", "#0000A0"]

        col_up_occ = list(cols.to_rgb(colors[0]))
        col_up_unocc = [.5 + .5 * channel for channel in col_up_occ]
        col_down_occ = list(cols.to_rgb(colors[1]))
        col_down_unocc = [.5 + .5 * channel for channel in col_down_occ]
        
        (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc) = self.spin_and_occupation_resolved_DOS(wavecar_object, *args, **kwargs)
        
        fig, ax = plt.subplots()
        fig.set_size_inches(3, 4.6)
        ax.fill_betweenx(LDOS_up_occ[0], LDOS_up_occ[1], color = col_up_occ)
        ax.fill_betweenx(LDOS_up_unocc[0], LDOS_up_unocc[1], color = col_up_unocc)
        ax.fill_betweenx(LDOS_down_occ[0], -LDOS_down_occ[1], color = col_down_occ)
        ax.fill_betweenx(LDOS_down_unocc[0], -LDOS_down_unocc[1], color = col_down_unocc)
        
        ax.set_xlabel("DOS up (a.u.)    DOS down (a.u.)")
        ax.set_ylabel("energy (eV)")
        ax.set_xticks([])
        
        en_range = kwargs.get("energy_range")
        ax.set_ylim(en_range[0], en_range[1])
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(.1))
        
        ax.grid(True, which = "both", axis = "y", color = "gray", linewidth = 0.5, alpha = 0.5)
        return fig

    def structure_plot(self, struc: Structure, max_bond_length: float = None, width: int = 800, height: int = 600, atom_size: float = .3, bond_size: float = .22, camera_type: str = "orthographic") -> nv.NGLWidget:
        atoms = ase.AseAtomsAdaptor.get_atoms(struc)
        Z = list(struc.atomic_numbers)
        R = struc.cart_coords
        Zcolors = [.5 * rgb if atomic_number > 0 else (0, 0, 0) for atomic_number, rgb in enumerate(jmol_colors)]
        Zcolors[6] = "#404040"
        
        if not max_bond_length:
            if 74 in Z: max_bond_length = 2.6 # Shortcut for working with TMDs
            else: max_bond_length = 1.6 # Shortcut fallback for organic stuff

        cutoffs = [max_bond_length / 2.0] * len(atoms)
        nl = NeighborList(cutoffs, skin = 0.0, bothways = True, self_interaction = False)
        nl.update(atoms)

        view = nv.show_ase(atoms)
        view.stage.set_parameters(depth_of_field = 0, fog_near = 100, fog_far = 100, camera_type = camera_type)
        
        bonds = []
        for atom_index in range(len(atoms)):
            Z1 = Z[atom_index]
            R1 = R[atom_index]
            color1 = Zcolors[Z1]
            
            neighbor_indices, offsets = nl.get_neighbors(atom_index)
            for (neighbor_index, offset) in zip(neighbor_indices, offsets):
                if neighbor_index < atom_index or np.any(offset != 0): continue
                
                Z2 = Z[neighbor_index]
                R2 = R[neighbor_index]
                bond_color = color1 + Zcolors[Z2]
                
                bonds.append(["cylinder", R1, R2, bond_color, bond_size, "bond"])
        view._add_shape(bonds)
        
        view.clear_representations()
        view.component_0.add_spacefill(radiusType = "vdw", radiusScale = atom_size)
        view.component_0.add_spacefill(selection = "_C", radiusType = "vdw", radiusScale = atom_size, colorValue = Zcolors[6])
        view.component_0.add_spacefill(selection = "_N", radiusType = "vdw", radiusScale = atom_size + .1, colorValue = Zcolors[8]) # Emphasize nitrogen
        view.control.center(np.mean(struc.cart_coords, axis = 0))

        view.height = f"{height}px"
        view.width = f"{width}px"
        view.layout.height = f"{height}px"
        view.layout.width = f"{width}px"
        return view

    def orbital_plot(self, wavecar_object: vaspwfc, ispin: int = 1, ikpt: int = 1, iband: int = 1, isolevel: float = .1, opacity: float = 1., flip_x: bool = False, flip_y: bool = False, flip_z: bool = False, upsampling: int = 1,
                     struc: Structure = None, max_bond_length: float = 2.6, atom_size: float = .3, bond_size: float = .22, struc_opacity: float = 1.,
                     width: int = 800, height: int = 600, camera_type: str = "orthographic") -> nv.NGLWidget:
        if not isinstance(wavecar_object, vaspwfc):
            print(f"Invalid wave function")
            return
        if not isinstance(opacity, float | int) or opacity < 0 or opacity > 1: opacity = 1.
        if not isinstance(struc_opacity, float | int) or struc_opacity < 0 or struc_opacity > 1: struc_opacity = 1.
        
        try:
            psi: np.ndarray = zoom(wavecar_object.wfc_r(ispin = ispin, ikpt = ikpt, iband = iband), zoom = upsampling, order = 3)
            if flip_x: psi = np.flip(psi, axis = 0)
            if flip_y: psi = np.flip(psi, axis = 1)
            if flip_z: psi = np.flip(psi, axis = 2)
            orb_plus = np.abs(np.clip(psi, a_min = 0, a_max = np.inf)) ** 2
            orb_minus = np.abs(np.clip(psi, a_min = -np.inf, a_max = 0) ** 2)
        
            cell_size_Ang = wavecar_object._Acell
            voxels = wavecar_object._ngrid * 2 * upsampling
            
            voxel_size = np.diag(cell_size_Ang) / voxels
        except Exception as e:
            print(f"{e}")
            return
        
        if isinstance(struc, Structure):
            view = self.structure_plot(struc, max_bond_length, width, height, atom_size, bond_size, camera_type)
            view.update_representation(component = len(view._ngl_component_names) - 2, repr_index = 0, opacity = struc_opacity, transparent = True, depthWrite = False)
            view.update_representation(component = len(view._ngl_component_names) - 1, repr_index = 0, opacity = struc_opacity, transparent = True, depthWrite = False)
        else:
            view = nv.NGLWidget()
        
        try:
            for orb, color in zip([orb_plus, orb_minus], [[.8, .4, 0], [0, .2, .9]]):
                (verts, faces, normals, values) = measure.marching_cubes(orb, level = isolevel * np.max(orb_plus))
                verts_Ang = verts * voxel_size
                
                v0 = verts_Ang[faces[:, 0]]
                v1 = verts_Ang[faces[:, 1]]
                v2 = verts_Ang[faces[:, 2]]
                face_normals = np.cross(v1 - v0, v2 - v0)
                norms = np.linalg.norm(face_normals, axis = 1, keepdims = True)
                face_normals = np.divide(face_normals, norms, out = np.zeros_like(face_normals), where = norms != 0)
                flat_normals = np.repeat(face_normals, 3, axis = 0).ravel().tolist()
                 
                flat_positions = verts_Ang[faces].ravel().tolist()
                num_mesh_vertices = faces.size
                flat_colors = color * num_mesh_vertices
                
                view.shape.add_mesh(flat_positions, flat_colors, None, flat_normals, "Isosurface")
                view.update_representation(component = len(view._ngl_component_names) - 1, repr_index = 0, side = "front", opacity = opacity, transparent = True, flatShading = False, depthWrite = True, opaqueBack = True)
        except:
            print("Problem creating the mesh")

        return view

    def LDOS_maps(self, wavecar_object: vaspwfc, struc: Structure, energy_values_meV: list | np.ndarray = [], z_values_pm: list | np.ndarray | float | int = 0.,
                width_values_pm: list | np.ndarray | float | int = 0., gamma_meV: float = 50, n_gammas: int = 5, output_folder: str = "LDOS_maps"):
        # Initialize important parameters
        gamma_eV = gamma_meV / 1000
        gamma2 = gamma_eV ** 2
        voxels = wavecar_object._ngrid * 2
        voxel_size_Ang = np.diag(wavecar_object._Acell) / voxels
        px_per_pm = 1 / (100 * np.mean(voxel_size_Ang))
        atom_z_values_nm = struc.cart_coords[:, 2] * .1
        z_surface_nm = np.mean(np.partition(atom_z_values_nm, -12)[-12:-10])
        z_nm_per_vox = voxel_size_Ang[2] / 10
        n_spins = int(wavecar_object._nspin)
        n_kpts = int(wavecar_object._nkpts)
        
        # Get the band energies and take a selection ranging from n_gammas times the Lorentzian width below the minimum energy value to n_gammas times above the maximum energy value
        energy_dict = self.get_eigenenergies_from_wavecar(wavecar_object)
        spin_up_energies = energy_dict["energies"]["spin up"]
        spin_down_energies = energy_dict["energies"]["spin down"]
        k_resolved_spin_up_energies = spin_up_energies.reshape(n_kpts, -1)
        k_resolved_spin_down_energies = spin_down_energies.reshape(n_kpts, -1)        
        
        min_up_index = min([int(np.where(k_resolved_spin_up_energies[kpt] > .001 * np.min(energy_values_meV) - n_gammas * (gamma_meV / 1000))[0][0]) for kpt in range(len(k_resolved_spin_up_energies))])
        min_down_index = min([int(np.where(k_resolved_spin_down_energies[kpt] > .001 * np.min(energy_values_meV) - n_gammas * (gamma_meV / 1000))[0][0]) for kpt in range(len(k_resolved_spin_down_energies))])
        min_orbital_index = min((min_up_index, min_down_index))
        max_up_index = max([int(np.where(k_resolved_spin_up_energies[kpt] < .001 * np.max(energy_values_meV) + n_gammas * (gamma_meV / 1000))[0][-1]) for kpt in range(len(k_resolved_spin_up_energies))])
        max_down_index = max([int(np.where(k_resolved_spin_down_energies[kpt] < .001 * np.max(energy_values_meV) + n_gammas * (gamma_meV / 1000))[0][-1]) for kpt in range(len(k_resolved_spin_down_energies))])
        max_orbital_index = max((max_up_index, max_down_index))
        orbital_indices = np.arange(min_orbital_index, max_orbital_index + 1, 1, dtype = np.int32)

        selected_spin_up_energies = np.concatenate([k_resolved_spin_up_energies[kpt][orbital_indices] for kpt in range(n_kpts)])
        selected_spin_down_energies = np.concatenate([k_resolved_spin_down_energies[kpt][orbital_indices] for kpt in range(n_kpts)])
        energies = np.concatenate((selected_spin_up_energies, selected_spin_down_energies))
        
        # Extract a subset of the wavefunctions from the wavecar file and store it in wfns
        wfns = np.zeros((n_spins, n_kpts, len(orbital_indices), voxels[0], voxels[1], voxels[2]), dtype = np.complex64)
        for spin_index in range(n_spins):
            for k_index in range(n_kpts):
                for index, orb_index in enumerate(orbital_indices):
                    wfns[spin_index, k_index, index] = wavecar_object.wfc_r(spin_index + 1, k_index + 1, orb_index + 1)
        
        # Create the output directory and clean the tip height and width
        os.makedirs(os.path.join(os.curdir, output_folder), exist_ok = True)
        if isinstance(z_values_pm, int | float): z_values_pm = [z_values_pm]
        if not isinstance(z_values_pm, list | np.ndarray): print("Invalid height value(s)")
        if isinstance(width_values_pm, int | float): width_values_pm = [width_values_pm]
        if not isinstance(width_values_pm, list | np.ndarray): print("Invalid width value(s)")
        if isinstance(energy_values_meV, int | float): energy_values_meV = [energy_values_meV]
        if not isinstance(energy_values_meV, list | np.ndarray): print("Invalid energy value(s)")
        if not isinstance(energy_values_meV, list | np.ndarray) or not isinstance(width_values_pm, list | np.ndarray) or not isinstance(z_values_pm, list | np.ndarray): return
                
        # Loop over heights
        for z_slice_height_pm in z_values_pm:
            # Slice out the 2D wavefunction from the 3D wavefunction at the requested height
            z_target = z_surface_nm + z_slice_height_pm / 1000
            z_slice_index = int(z_target / z_nm_per_vox)
            wfns2D = wfns[:, :, :, :, :, z_slice_index]            
            spin_k_collapsed_wfns2D = wfns2D.reshape(-1, voxels[0], voxels[1]) # Flatten out the k and spin
            
            for width_pm in width_values_pm:
                # Broaden the wavefunction according to their overlap with the Gaussian tip wavefunction
                width_px = width_pm * px_per_pm
                broadened_wfns2D = [gaussian_filter(image, width_px, mode = "wrap") for image in spin_k_collapsed_wfns2D]
                densities = np.asarray(np.abs(np.array(broadened_wfns2D)) ** 2, dtype = np.float32)

                for target_energy_meV in energy_values_meV:
                    en_differences = np.array(energies, dtype = np.float32) - (.001 * target_energy_meV)
                    
                    weights = gamma_eV / (gamma2 + en_differences ** 2)
                    weights /= np.sum(weights)

                    image = np.average(densities, axis = 0, weights = weights)
                    plt.imsave(f"./{output_folder}/LDOS_w{int(round(width_pm))}pm_h{int(z_slice_height_pm)}pm@{int(round(target_energy_meV))}meV.png", image, cmap = "gray")                    
        return

initio = Initio()

In [4]:
folders = ["C:\\DFT\\WS2", "C:\\DFT\\WSe2", "C:\\DFT\\Heteroatoms\\7-AGNR_defects"]
calculations = ["WS2\\primitive", "Vac_S_0", "WS2\\supercell", "Co_S_0", "Co_S_-1", "Co_S_+1", "Eu_on_S_0", "Eu_on_S_1", "Eu_on_S_2", "Co_Se_0_PBE0", "5-AGNR_3_N_row2", "5-AGNR_12_N_row2", "5-AGNR_12_N_row3", "7-AGNR_8_N_row2"]

In [34]:
calculation_name = "Co_S_0"
target_folder = find_folder(calculation_name, base_folder = "C:\\DFT")
os.chdir(target_folder)
print(f"Analyzing files in calculation folder {os.getcwd()}")

[struc, wfc] = [initio.read_vasp_file(file_name) for file_name in ["./CONTCAR", "./WAVECAR"]]

Analyzing files in calculation folder C:\DFT\WS2\Co_S_0


In [ ]:
view = initio.structure_plot(struc, width = 1000, height = 800, camera_type = "perspective")
saved_orientation = view._camera_orientation

view

In [74]:
view.control.spin([1, 0, 0], np.deg2rad(30))
#view.control.spin([0, 0, 1], np.deg2rad(180))

In [75]:
view.download_image(filename = f"./{calculation_name}_struc_perspective.png")

In [ ]:
HL_dict = initio.get_HOMO_LUMO(wfc)
[HOMO_up_index, HOMO_down_index] = [HL_dict.get(attribute) for attribute in ["HOMO_up_index", "HOMO_down_index"]]
print(f"Spin up HOMO / LUMO band indices: [{HOMO_up_index}, {HOMO_up_index + 1}]\nSpin down HOMO / LUMO band indices: [{HOMO_down_index}, {HOMO_down_index + 1}]")

gamma = .04
gamma2 = gamma ** 2
figure = initio.DOS_plot(wfc, energy_range = [-3.6, 0], dE = .01, gamma = gamma, colors = ["#0000A0", "#A00000"])
plt.tight_layout()
plt.savefig(f"./{calculation_name}_DOS_gamma_{int(1000 * gamma)}_meV.svg")
plt.show()

In [453]:
initio.LDOS_maps(wfc, struc, energy_values_meV = [-3900, -3700, -3500, -3400, -2800, -1800, -1500, -1100], z_values_pm = [100, 150, 200], width_values_pm = [0.0, 50.0, 100.0, 150.0, 200.0], gamma_meV = 100, output_folder = "LDOS_maps")

In [ ]:
energy_dict = initio.get_eigenenergies_from_wavecar(wfc)
np.where(energy_dict["energies"]["spin up"] > -2)

In [96]:
view = initio.orbital_plot(wfc, 1, 1, 627, isolevel = .05, opacity = .8, upsampling = 2, struc = struc, max_bond_length = 2.6, atom_size = .2, bond_size = .1, width = 1000, height = 1000)
view

NGLWidget(layout=Layout(height='1000px', width='1000px'))

In [73]:
view.control.spin([1,0,0], np.deg2rad(20))
#view.control.spin([0, 0, 1], np.deg2rad(90))

In [74]:
view.download_image()

In [ ]:
cell_size_Ang = wfc._Acell
cell_size_nm = cell_size_Ang * .1
frame_dict = {"domain (nm)": np.array([cell_size_nm[0, 0], cell_size_nm[1, 1], cell_size_nm[2, 2]], dtype = np.float32)}
w_nm = frame_dict["domain (nm)"][0]
h_nm = frame_dict["domain (nm)"][1]
z_nm = frame_dict["domain (nm)"][2]

voxels = wfc._ngrid * 2
spin_index = 2
k_index = 1

x_values = np.linspace(0, w_nm, voxels[0])
y_values = np.linspace(0, h_nm, voxels[1])
z_values = np.linspace(0, z_nm, voxels[2])
orbital_indices = np.arange(HOMO_up_index - 16, HOMO_up_index + 15)
spins = np.array(["spin up", "spin down"], dtype = np.str_)
n_spins = int(wfc._nspin)
n_kpts = int(wfc._nkpts)

energy_dict = initio.get_eigenenergies_from_wavecar(wfc)
spin_up_energies = energy_dict["energies"]["spin up"][orbital_indices]
spin_down_energies = energy_dict["energies"]["spin down"][orbital_indices]
spin_up_occupations = energy_dict["occupations"]["spin up"][orbital_indices]
spin_down_occupations = energy_dict["occupations"]["spin down"][orbital_indices]

z_slice_height = 0.3

atom_z_values_nm = struc.cart_coords[:, 2] * .1
z_surface = np.mean(np.partition(atom_z_values_nm, -10)[-10:])
z_nm_per_vox = z_nm / voxels[2]

z_target = z_surface + z_slice_height
z_slice_index = int(z_target / z_nm_per_vox)

with h5py.File(f"./{calculation_name}_wfns@{int(z_slice_height * 10)}A.h5", "w") as f:
    frame_group = f.create_group("frame")
    frame_group.attrs.update({key: value for key, value in frame_dict.items()})
    
    main_group = f.create_group("wavefunctions")
    main_group.create_dataset("wavefunctions", data = wfns[:, :, :, :, :, z_slice_index], dtype = np.complex64)    
    main_group.create_dataset("spin", data = np.arange(wfns.shape[0], dtype = np.uint32), dtype = np.uint32)
    main_group.create_dataset("k-point", data = np.arange(wfns.shape[1], dtype = np.uint32), dtype = np.uint32)
    main_group.create_dataset("orbital", data = orbital_indices, dtype = np.uint32)
    main_group.create_dataset("x (nm)", data = x_values, dtype = np.float32)
    main_group.create_dataset("y (nm)", data = y_values, dtype = np.float32)
    
    main_group.attrs.update({"NX_class": "NXdata", "signal": "wavefunctions", "axes": ["spin", "k-point", "orbital", "x (nm)", "y (nm)"]})
    
    lattice_group = f.create_group("lattice")
    lattice_group.create_dataset("lattice_vectors (nm)", data = (wfc._Acell) * .1, dtype = np.float32)
    lattice_group.create_dataset("reciprocal_vectors (nm-1)", data = (wfc._Bcell) * 10, dtype = np.float32)
    
    energy_group = f.create_group("energy")
    energy_group.create_dataset("orbital_energies", data = np.stack([spin_up_energies, spin_down_energies]))
    energy_group.create_dataset("orbital_occupations", data = np.stack([spin_up_occupations, spin_down_occupations]))
    energy_group.create_dataset("spin", data = np.array([0, 1], dtype = np.uint32), dtype = np.uint32)